# Day 1 — Exploratory Data Analysis: UCI Credit Card Default

**Goal:** Understand the dataset structure, target distribution, feature types, and key patterns before any modeling.

**Dataset:** [UCI Default of Credit Card Clients](https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients)  
**Target:** `default.payment.next.month` (1 = default, 0 = no default)

---

## Sections
1. Load & Inspect
2. Target Distribution (Class Imbalance)
3. Demographic Features
4. Payment History Features
5. Bill & Payment Amount Features
6. Correlation Analysis
7. Key Observations & Next Steps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded.')

## 1. Load & Inspect

Download the dataset from UCI and place the `.xls` file at `data/raw/UCI_Credit_Card.xls`.

Direct download:  
```
https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls
```

In [ ]:
import requests
from pathlib import Path

RAW_PATH = Path('../data/raw/UCI_Credit_Card.xls')

if not RAW_PATH.exists():
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
    print('Downloading dataset...')
    r = requests.get(url, timeout=30)
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    RAW_PATH.write_bytes(r.content)
    print(f'Saved to {RAW_PATH}')
else:
    print(f'Dataset already exists at {RAW_PATH}')

In [ ]:
# The UCI file has a header row on row 0 and actual column names on row 1
df = pd.read_excel(RAW_PATH, header=1, index_col=0)

# Rename target column for clarity
df = df.rename(columns={'default payment next month': 'default'})

print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None — clean dataset!')

In [ ]:
df.describe()

## 2. Target Distribution — Class Imbalance

> **Interview talking point:** Class imbalance is the norm in credit default data. ~22% default rate means accuracy is a misleading metric — always report AUC-ROC, Gini, and KS statistic.

In [ ]:
target_counts = df['default'].value_counts()
target_pct    = df['default'].value_counts(normalize=True) * 100

print('Class Distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percent': target_pct.round(1)}))

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No Default (0)', 'Default (1)'], target_counts, color=['#2196F3', '#F44336'])
for bar, pct in zip(bars, target_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', fontweight='bold')
ax.set_title('Target Distribution', fontsize=14)
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('../reports/figures/01_target_distribution.png', dpi=150)
plt.show()

## 3. Demographic Features

**Columns:** `LIMIT_BAL`, `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`

**Codebook:**
- `SEX`: 1=male, 2=female
- `EDUCATION`: 1=graduate, 2=university, 3=high school, 4=others
- `MARRIAGE`: 1=married, 2=single, 3=others

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Credit limit distribution by default
df.groupby('default')['LIMIT_BAL'].plot(kind='kde', ax=axes[0,0], legend=True)
axes[0,0].set_title('Credit Limit by Default Status')
axes[0,0].set_xlabel('Credit Limit (NTD)')

# Default rate by SEX
sex_default = df.groupby('SEX')['default'].mean() * 100
sex_default.index = ['Male', 'Female']
sex_default.plot(kind='bar', ax=axes[0,1], color='#FF9800', rot=0)
axes[0,1].set_title('Default Rate by Gender (%)')
axes[0,1].set_ylabel('Default Rate (%)')

# Default rate by EDUCATION
edu_default = df.groupby('EDUCATION')['default'].mean() * 100
edu_default.index = ['Grad School', 'University', 'High School', 'Others', '?', '?']
edu_default.plot(kind='bar', ax=axes[0,2], color='#9C27B0', rot=30)
axes[0,2].set_title('Default Rate by Education (%)')

# Default rate by MARRIAGE
mar_default = df.groupby('MARRIAGE')['default'].mean() * 100
mar_default.index = ['?', 'Married', 'Single', 'Others']
mar_default.plot(kind='bar', ax=axes[1,0], color='#4CAF50', rot=30)
axes[1,0].set_title('Default Rate by Marital Status (%)')

# AGE distribution by default
df[df.default==0]['AGE'].plot(kind='hist', bins=30, ax=axes[1,1], alpha=0.6, label='No Default', color='#2196F3')
df[df.default==1]['AGE'].plot(kind='hist', bins=30, ax=axes[1,1], alpha=0.6, label='Default', color='#F44336')
axes[1,1].set_title('Age Distribution by Default Status')
axes[1,1].legend()

# Default rate by age group
df['age_group'] = pd.cut(df['AGE'], bins=[20, 30, 40, 50, 60, 80], labels=['20s', '30s', '40s', '50s', '60+'])
age_default = df.groupby('age_group', observed=True)['default'].mean() * 100
age_default.plot(kind='bar', ax=axes[1,2], color='#00BCD4', rot=0)
axes[1,2].set_title('Default Rate by Age Group (%)')

plt.suptitle('Demographic Features vs Default', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/02_demographic_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Payment History Features (PAY_0 through PAY_6)

**Codebook:** -1=paid duly, 0=minimum paid, 1=one month delay, 2=two months, ...

> **Key insight:** Repayment status in recent months is typically the strongest predictor of default — this is your feature engineering target for Day 3.

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

months = ['Sep 2005', 'Aug 2005', 'Jul 2005', 'Jun 2005', 'May 2005', 'Apr 2005']

for i, (col, month) in enumerate(zip(pay_cols, months)):
    pay_default = df.groupby(col)['default'].mean() * 100
    pay_default.plot(kind='bar', ax=axes[i], color='#FF5722', rot=0)
    axes[i].set_title(f'{col} ({month}) Default Rate')
    axes[i].set_xlabel('Payment Status')
    axes[i].set_ylabel('Default Rate (%)')

plt.suptitle('Payment Status vs Default Rate (by Month)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_payment_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation of payment status with default
print('\nCorrelation with default:')
print(df[pay_cols + ['default']].corr()['default'].sort_values(ascending=False))

## 5. Bill & Payment Amount Features

In [ ]:
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

# Median bill vs payment by default status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('default')[bill_cols].median().T.plot(ax=axes[0])
axes[0].set_title('Median Bill Amount by Month & Default Status')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Median Bill (NTD)')
axes[0].legend(['No Default', 'Default'])

df.groupby('default')[pay_amt_cols].median().T.plot(ax=axes[1])
axes[1].set_title('Median Payment Amount by Month & Default Status')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Median Payment (NTD)')
axes[1].legend(['No Default', 'Default'])

plt.tight_layout()
plt.savefig('../reports/figures/04_bill_payment_amounts.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
# Top features correlated with default
corr_with_default = df.select_dtypes(include='number').corr()['default'].drop('default').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_with_default.head(15).plot(kind='barh', ax=axes[0], color='#3F51B5')
axes[0].set_title('Top 15 Features by |Correlation| with Default', fontsize=12)
axes[0].set_xlabel('|Correlation|')
axes[0].invert_yaxis()

# Full correlation heatmap (numeric only)
key_cols = pay_cols + ['LIMIT_BAL', 'AGE', 'default']
sns.heatmap(df[key_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Correlation Heatmap — Payment History + Key Features', fontsize=12)

plt.tight_layout()
plt.savefig('../reports/figures/05_correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features correlated with default:')
print(corr_with_default.head(10))

## 7. Key Observations & Next Steps

### What We Learned Today

| Observation | Implication for Modeling |
|-------------|-------------------------|
| ~22% default rate | Class imbalance — use `class_weight='balanced'` or SMOTE |
| PAY_0 strongest predictor | Recent payment behavior is the #1 signal |
| Defaulters pay significantly less | Payment ratio feature will be highly predictive |
| EDUCATION categories 0/5/6 are undocumented | Bin with category 4 (others) |
| MARRIAGE category 0 is undocumented | Bin with category 3 (others) |
| No missing values | Clean dataset — can focus on feature engineering |

### Day 2 → Feature Engineering Ideas

```python
# Ideas to implement tomorrow:
# 1. Payment ratio: PAY_AMT / BILL_AMT (how much of bill was paid)
# 2. Credit utilization: BILL_AMT1 / LIMIT_BAL
# 3. Delinquency streak: consecutive months with PAY > 0
# 4. Payment trend: is payment amount increasing or decreasing?
# 5. Max delinquency: max(PAY_0..PAY_6)
```

### Interview Talking Points from Today's EDA

- **On class imbalance:** "The ~22% default rate means a naive model predicting 'no default' achieves 78% accuracy — that's why I use AUC-ROC and Gini coefficient, which are threshold-independent."
- **On PAY_0 dominance:** "Recent payment behavior dominates because it directly signals current financial stress. This aligns with behavioral scoring models used in industry."
- **On data cleaning:** "Even on a 'clean' dataset, I found undocumented categories in EDUCATION and MARRIAGE that need binning — real-world data is always messier."

In [ ]:
# Save processed base dataframe for Day 2
df.drop(columns=['age_group'], errors='ignore').to_parquet('../data/processed/credit_card_base.parquet', index=True)
print('Saved base dataset to data/processed/credit_card_base.parquet')
print(f'Final shape: {df.shape}')